In [272]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import json
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import precision_score, recall_score, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

In [273]:
data = json.load(open("data/pokemon-sets.json"))

In [274]:
rows = []
cardSkipCount = 0
entrySkipCount = 0

for seriesId, series in data.items():
    for card in series["cards"]:
        if len(card["priceEvolutionCM"]) == 0 or len(card["priceEvolutionTCG"]) == 0:
            cardSkipCount += 1
            print(f"Skipping card {card['name']} from series {seriesId} because it has no initial price in cardmarket data")
            continue

        currentPriceCM = None
        i = 0
        while i < len(card["priceEvolutionCM"]):
            if "price" in card["priceEvolutionCM"][i]:
                currentPriceCM = card["priceEvolutionCM"][i]["price"]
                break

            i +=1

        if currentPriceCM is None:
            cardSkipCount += 1
            print(f"Skipping card {card['name']} from series {seriesId} because it has no initial price in cardmarket data")
            continue


        currentPriceTCG = None
        j = 0
        while j < len(card["priceEvolutionTCG"]):
            if "price" in card["priceEvolutionTCG"][j]:
                currentPriceTCG = card["priceEvolutionTCG"][j]["price"]
                break

            j += 1

        if currentPriceTCG is None:
            cardSkipCount += 1
            print(f"Skipping card {card['name']} from series {seriesId} because it has no initial price in tcgplayer data")
            continue

        t = 0
        for cardmarketdata, tcgplayerdata in zip(card["priceEvolutionCM"], card["priceEvolutionTCG"]):
            currentPriceCM = cardmarketdata["price"] if "price" in cardmarketdata else currentPriceCM
            currentPriceTCG = tcgplayerdata["price"] if "price" in tcgplayerdata else currentPriceTCG
            rows.append({
                "t": t,
                "seriesId": seriesId,
                "commercial_name": series["commercialName"],
                "hype_level": series["hypeLevel"],
                "age_level": series["ageLevel"],

                "card_id": card["id"],
                "name": card["name"],
                "number": card["number"],
                "rarity": card["rarity"],
                "pokedex_id": card["pokedexId"],
                "image": card["image"],

                "current_offer": sum(list(map(lambda x: x["number"], card["offer"]))),
                "current_demand": sum(list(map(lambda x: x["number"], card["demand"]))),

                "cardmarket_price": currentPriceCM,
                "tcgplayer_price": currentPriceTCG
            })

            t += 1

df = pd.DataFrame(rows)
print(f"Dataframe shape: {df.shape} with {cardSkipCount} skipped cards and {entrySkipCount} skipped entries")
df.head()

Skipping card Salarsen from series PFL because it has no initial price in cardmarket data
Skipping card Reshiram de N from series JTG because it has no initial price in cardmarket data
Dataframe shape: (173342, 15) with 2 skipped cards and 0 skipped entries


,t,seriesId,commercial_name,hype_level,age_level,card_id,name,number,rarity,pokedex_id,image,current_offer,current_demand,cardmarket_price,tcgplayer_price
0,0,PBL,ME05,1,1,26312,Mimantis,85,2,753.0,https://pokecardex-scans.b-cdn.net/sets/PBL/FR...,20,153,8.0,4.82
1,1,PBL,ME05,1,1,26312,Mimantis,85,2,753.0,https://pokecardex-scans.b-cdn.net/sets/PBL/FR...,20,153,7.9,4.98
2,2,PBL,ME05,1,1,26312,Mimantis,85,2,753.0,https://pokecardex-scans.b-cdn.net/sets/PBL/FR...,20,153,7.9,4.47
3,3,PBL,ME05,1,1,26312,Mimantis,85,2,753.0,https://pokecardex-scans.b-cdn.net/sets/PBL/FR...,20,153,7.9,5.23
4,4,PBL,ME05,1,1,26312,Mimantis,85,2,753.0,https://pokecardex-scans.b-cdn.net/sets/PBL/FR...,20,153,7.9,4.46


In [275]:
def extract_features_at_T(df_source, T):
    """Engineers features using only data up to day T."""
    df_hist = df_source[df_source['t'] <= T].copy()
    df_hist = df_hist.sort_values(['card_id', 't'])
    df_hist['daily_return'] = df_hist.groupby('card_id')['cardmarket_price'].pct_change()

    def _extract_single(group):
        latest = group.iloc[-1]
        price_t = latest['cardmarket_price']
        tcg_price_t = latest['tcgplayer_price']
        
        def safe_get_price(days_back):
            target_t = T - days_back
            sub = group[group['t'] == target_t]['cardmarket_price']
            return sub.values[0] if len(sub) > 0 else group['cardmarket_price'].iloc[0]

        price_7 = safe_get_price(7)
        price_14 = safe_get_price(14)
        price_30 = safe_get_price(30)
        
        return pd.Series({
            'card_id': latest['card_id'],
            'seriesId': latest['seriesId'],
            'rarity': latest['rarity'],
            'hype_level': latest['hype_level'],
            'age_level': latest['age_level'],
            'current_offer': latest['current_offer'],
            'current_demand': latest['current_demand'],
            'demand_to_offer_ratio': latest['current_demand'] / (latest['current_offer'] + 1e-5),
            'price_T': price_t,
            'cross_market_ratio': price_t / (tcg_price_t + 1e-5),
            'return_7d': (price_t - price_7) / price_7,
            'return_14d': (price_t - price_14) / price_14,
            'return_30d': (price_t - price_30) / price_30,
            'volatility_14d': group.tail(14)['daily_return'].std(),
            'volatility_30d': group.tail(30)['daily_return'].std(),
            'sma_7_ratio': price_t / group.tail(7)['cardmarket_price'].mean(),
            'sma_30_ratio': price_t / group.tail(30)['cardmarket_price'].mean(),
        })

    return df_hist.groupby('card_id').apply(_extract_single).reset_index(drop=True)

In [276]:
all_cards = df['card_id'].unique()

holdout_fraction = 0.15
n_holdout = int(len(all_cards) * holdout_fraction)
holdout_cards = np.random.choice(all_cards, size=n_holdout, replace=False)

df_holdout = df[df['card_id'].isin(holdout_cards)].copy()
df_train_pool = df[~df['card_id'].isin(holdout_cards)].copy()

print(f"Total Unique Cards: {len(all_cards)}")
print(f"Reserved for Simulation: {len(holdout_cards)} cards")
print(f"Available for CV:        {len(df_train_pool['card_id'].unique())} cards\n")

Total Unique Cards: 1162
Reserved for Simulation: 174 cards
Available for CV:        988 cards



In [277]:
def run_logo_experiment(df_source, T, X=14, target_strong=0.85, target_buy=0.75):
    """
    Runs Leave-One-Group-Out CV for a given T and horizon X.
    Calculates out-of-fold probabilities and aggregates group-level precision metrics.
    """
    # 1. Label Generation
    price_T = df_source[df_source['t'] == T].set_index('card_id')['cardmarket_price']
    price_TX = df_source[df_source['t'] == (T + X)].set_index('card_id')['cardmarket_price']
    
    valid_cards = price_T.index.intersection(price_TX.index)
    labels = ((price_TX.loc[valid_cards] - price_T.loc[valid_cards]) > 0).astype(int).rename('target').reset_index()

    # 2. Extract Features up to day T
    features_df = extract_features_at_T(df_source[df_source['card_id'].isin(valid_cards)], T)
    dataset = pd.merge(features_df, labels, on='card_id')

    cat_cols = ['seriesId', 'rarity']
    for col in cat_cols:
        dataset[col] = dataset[col].astype('category')

    feature_cols = [c for c in dataset.columns if c not in ['card_id', 'target']]
    X_data = dataset[feature_cols]
    y_data = dataset['target']
    groups = dataset['card_id']

    # 3. Leave-One-Group-Out Validation Loop
    logo = LeaveOneGroupOut()
    oof_preds = np.zeros(len(dataset))
    
    params = {
        'objective': 'binary',
        'metric': 'auc',
        'learning_rate': 0.02,
        'num_leaves': 10,
        'min_child_samples': 15,
        'colsample_bytree': 0.8,
        'subsample': 0.8,
        'verbose': -1
    }

    # Group-level prediction tracking
    group_precisions_at_05 = []

    for train_idx, val_idx in logo.split(X_data, y_data, groups=groups):
        X_train, y_train = X_data.iloc[train_idx], y_data.iloc[train_idx]
        X_val, y_val = X_data.iloc[val_idx], y_data.iloc[val_idx]

        train_data = lgb.Dataset(X_train, label=y_train)
        
        # LOGO single-instance validation set
        val_data = lgb.Dataset(X_val, label=y_val, reference=train_data)

        model = lgb.train(
            params,
            train_data,
            num_boost_round=150,
            valid_sets=[train_data],
            callbacks=[lgb.early_stopping(stopping_rounds=20, verbose=False)]
        )

        val_pred = model.predict(X_val, num_iteration=model.best_iteration)[0]
        oof_preds[val_idx] = val_pred
        
        # Compute group-level match (1 if correctly matched direction at 0.5, 0 otherwise)
        group_actual = y_val.values[0]
        group_pred_binary = int(val_pred >= 0.5)
        if group_pred_binary == 1:
            group_precisions_at_05.append(1.0 if group_actual == 1 else 0.0)

    # 4. Global Probability Threshold Search for Signal Tiers
    thresholds = np.linspace(0.40, 0.95, 111)
    
    thresh_strong = 0.95
    thresh_buy = 0.75
    
    for t in thresholds:
        preds_b = (oof_preds >= t).astype(int)
        if (preds_b == 1).sum() > 0:
            p = precision_score(y_data, preds_b, pos_label=1, zero_division=0)
            if p >= target_buy and thresh_buy == 0.75:
                thresh_buy = t
            if p >= target_strong:
                thresh_strong = t
                break

    # 5. Evaluate Group-Level Precision Distributions at Threshold Tiers
    dataset['oof_prob'] = oof_preds
    
    # Calculate Card-Level Precision (Group Level Aggregate Stats)
    card_results = []
    for cid, group_df in dataset.groupby('card_id'):
        actual = group_df['target'].values[0]
        prob = group_df['oof_prob'].values[0]
        
        signal = 'NO_BUY'
        if prob >= thresh_strong:
            signal = 'STRONG_BUY'
        elif prob >= thresh_buy:
            signal = 'BUY'
            
        card_results.append({
            'card_id': cid,
            'target': actual,
            'oof_prob': prob,
            'signal': signal
        })
    
    results_df = pd.DataFrame(card_results)
    
    # Group Aggregation Stats
    buy_cards = results_df[results_df['signal'].isin(['BUY', 'STRONG_BUY'])]
    strong_cards = results_df[results_df['signal'] == 'STRONG_BUY']
    
    buy_precision = buy_cards['target'].mean() if len(buy_cards) > 0 else 0.0
    strong_precision = strong_cards['target'].mean() if len(strong_cards) > 0 else 0.0
    
    # Group Precision Stats across out-of-fold instances where Buy was triggered
    g_precisions = np.array(group_precisions_at_05) if len(group_precisions_at_05) > 0 else np.array([0.0])

    return {
        'T': T,
        'AUC': roc_auc_score(y_data, oof_preds),
        'Thresh_Buy': thresh_buy,
        'Thresh_StrongBuy': thresh_strong,
        'Buy_Tier_Precision': buy_precision,
        'Buy_Count': len(buy_cards),
        'StrongBuy_Tier_Precision': strong_precision,
        'StrongBuy_Count': len(strong_cards),
        'Min_Group_Precision': np.min(g_precisions),
        'Median_Group_Precision': np.median(g_precisions),
        'Mean_Group_Precision': np.mean(g_precisions),
        'oof_df': dataset,
    }, model

In [278]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import optuna

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    roc_auc_score,
    precision_score
)
def objective(
    trial,
    X_train,
    y_train,
    X_val,
    y_val,
    target_buy=0.75,
    min_buy_predictions=10
):
    params = {
        'objective': 'binary',
        'metric': 'auc',

        'learning_rate': trial.suggest_float(
            'learning_rate', 0.01, 0.08, log=True
        ),

        'num_leaves': trial.suggest_int(
            'num_leaves', 5, 40
        ),

        'min_child_samples': trial.suggest_int(
            'min_child_samples', 5, 40
        ),

        'max_depth': trial.suggest_int(
            'max_depth', 3, 8
        ),

        'feature_fraction': trial.suggest_float(
            'feature_fraction', 0.6, 1.0
        ),

        'bagging_fraction': trial.suggest_float(
            'bagging_fraction', 0.6, 1.0
        ),

        'bagging_freq': trial.suggest_int(
            'bagging_freq', 0, 5
        ),

        'lambda_l1': trial.suggest_float(
            'lambda_l1', 1e-3, 10, log=True
        ),

        'lambda_l2': trial.suggest_float(
            'lambda_l2', 1e-3, 10, log=True
        ),

        'verbosity': -1,
        'random_state': 42
    }

    train_data = lgb.Dataset(
        X_train,
        label=y_train
    )

    val_data = lgb.Dataset(
        X_val,
        label=y_val,
        reference=train_data
    )

    model = lgb.train(
        params,
        train_data,
        num_boost_round=1000,
        valid_sets=[val_data],
        callbacks=[
            lgb.early_stopping(
                50,
                verbose=False
            )
        ]
    )

    pred = model.predict(
        X_val,
        num_iteration=model.best_iteration
    )

    buy_mask = pred >= target_buy

    n_buy = int(buy_mask.sum())

    if n_buy < min_buy_predictions:
        return 0.0

    precision = float(
        y_val[buy_mask].mean()
    )

    coverage = n_buy / len(y_val)

    score = precision + (0.05 * coverage)

    trial.set_user_attr(
        'precision',
        precision
    )

    trial.set_user_attr(
        'buy_count',
        n_buy
    )

    trial.set_user_attr(
        'coverage',
        coverage
    )

    trial.set_user_attr(
        'auc',
        roc_auc_score(y_val, pred)
    )

    trial.set_user_attr(
        'best_iteration',
        model.best_iteration
    )

    return score


def train(
    df_source,
    T,
    X=14,
    target_buy=0.75,
    validation_size=0.20,
    n_trials=100,
    min_buy_predictions=10,
    random_state=42
):
    price_T = (
        df_source[df_source['t'] == T]
        .set_index('card_id')['cardmarket_price']
    )

    price_TX = (
        df_source[df_source['t'] == (T + X)]
        .set_index('card_id')['cardmarket_price']
    )

    valid_cards = price_T.index.intersection(
        price_TX.index
    )

    if len(valid_cards) == 0:
        raise ValueError(
            f"No cards have both t={T} and t={T + X}."
        )

    labels = (
        (
            price_TX.loc[valid_cards]
            >
            price_T.loc[valid_cards]
        )
        .astype(int)
        .rename('target')
        .reset_index()
    )

    print(f"T = {T}")
    print(f"Horizon = {X}")
    print(f"Valid cards = {len(valid_cards)}")
    print(
        f"Positive rate = {labels['target'].mean():.3f}"
    )

    features_df = extract_features_at_T(
        df_source[
            df_source['card_id'].isin(valid_cards)
        ],
        T
    )

    dataset = pd.merge(
        features_df,
        labels,
        on='card_id',
        how='inner'
    )

    if dataset['card_id'].duplicated().any():

        print(
            "Warning: multiple rows per card detected."
        )

        dataset = (
            dataset
            .drop_duplicates(
                subset='card_id',
                keep='last'
            )
        )

    cat_cols = [
        'seriesId',
        'rarity'
    ]

    cat_cols = [
        col for col in cat_cols
        if col in dataset.columns
    ]

    for col in cat_cols:
        dataset[col] = dataset[col].astype('category')

    feature_cols = [
        c
        for c in dataset.columns
        if c not in [
            'card_id',
            'target'
        ]
    ]

    X_data = dataset[feature_cols]
    y_data = dataset['target']

    groups = dataset['card_id']

    splitter = GroupShuffleSplit(
        n_splits=1,
        test_size=validation_size,
        random_state=random_state
    )

    train_idx, val_idx = next(
        splitter.split(
            X_data,
            y_data,
            groups=groups
        )
    )

    X_train = X_data.iloc[train_idx]
    X_val = X_data.iloc[val_idx]

    y_train = y_data.iloc[train_idx]
    y_val = y_data.iloc[val_idx]

    train_cards = groups.iloc[train_idx].unique()
    val_cards = groups.iloc[val_idx].unique()

    print()
    print("Card-level split:")
    print(f"Training cards   = {len(train_cards)}")
    print(f"Validation cards = {len(val_cards)}")

    overlap = set(train_cards).intersection(
        set(val_cards)
    )

    if overlap:
        raise ValueError(
            f"Card leakage detected: {len(overlap)} cards overlap."
        )

    print(
        f"Training positive rate   = {y_train.mean():.3f}"
    )
    print(
        f"Validation positive rate = {y_val.mean():.3f}"
    )

    study = optuna.create_study(
        direction='maximize'
    )

    study.optimize(
        lambda trial: objective(
            trial,
            X_train,
            y_train,
            X_val,
            y_val,
            target_buy=target_buy,
            min_buy_predictions=min_buy_predictions
        ),
        n_trials=n_trials
    )

    best_trial = study.best_trial

    print()
    print("========== OPTUNA RESULT ==========")

    print(
        f"Best BUY score = "
        f"{best_trial.value:.4f}"
    )

    print(
        f"BUY predictions = "
        f"{best_trial.user_attrs.get('buy_count', 0)}"
    )

    print(
        f"Validation AUC = "
        f"{best_trial.user_attrs.get('auc', 0):.4f}"
    )

    print(
        f"Best iteration = "
        f"{best_trial.user_attrs.get('best_iteration', 0)}"
    )

    print()
    print("Best parameters:")

    for key, value in study.best_params.items():
        print(f"  {key}: {value}")

    final_params = study.best_params.copy()

    final_params.update({
        'objective': 'binary',
        'metric': 'auc',
        'verbosity': -1,
        'random_state': random_state
    })

    best_iteration = best_trial.user_attrs.get(
        'best_iteration',
        150
    )

    final_train_data = lgb.Dataset(
        X_data,
        label=y_data,
        categorical_feature=cat_cols
    )

    print()
    print(
        f"Training final model on ALL "
        f"{len(dataset)} cards..."
    )

    model = lgb.train(
        final_params,
        final_train_data,
        num_boost_round=best_iteration
    )

    print("Final model trained.")

    return model, feature_cols

In [279]:
T_setups = [60]
horizon_X = [7, 14,30, 60]
comparison_results = []
models = {}
print("Running Leave-One-Group-Out Validation across T Setups...\n")

for T_val in T_setups:
    for X_val in horizon_X:
        print(f"Running for T = {T_val}, X = {X_val}...")
        res, model = run_logo_experiment(df_train_pool, T=T_val, X=X_val)
        comparison_results.append(res)
        models[(T_val, X_val)] = model
        print(f"--- Completed T = {T_val}, X = {X_val} ---")
        print(f"  AUC:                    {res['AUC']:.4f}")
        print(f"  Mean Group Precision:   {res['Mean_Group_Precision']:.4f}")
        print(f"  Buy Signal Threshold:   >= {res['Thresh_Buy']:.3f}  (Precision: {res['Buy_Tier_Precision']:.2%}, Count: {res['Buy_Count']})")
        print(f"  Strong Buy Threshold:   >= {res['Thresh_StrongBuy']:.3f}  (Precision: {res['StrongBuy_Tier_Precision']:.2%}, Count: {res['StrongBuy_Count']})\n")

Running Leave-One-Group-Out Validation across T Setups...

Running for T = 60, X = 7...
--- Completed T = 60, X = 7 ---
  AUC:                    0.7272
  Mean Group Precision:   0.7204
  Buy Signal Threshold:   >= 0.595  (Precision: 75.00%, Count: 672)
  Strong Buy Threshold:   >= 0.815  (Precision: 85.08%, Count: 248)

Running for T = 60, X = 14...
--- Completed T = 60, X = 14 ---
  AUC:                    0.7581
  Mean Group Precision:   0.7714
  Buy Signal Threshold:   >= 0.400  (Precision: 75.70%, Count: 852)
  Strong Buy Threshold:   >= 0.780  (Precision: 85.06%, Count: 435)

Running for T = 60, X = 30...
--- Completed T = 60, X = 30 ---
  AUC:                    0.8137
  Mean Group Precision:   0.8156
  Buy Signal Threshold:   >= 0.400  (Precision: 79.88%, Count: 855)
  Strong Buy Threshold:   >= 0.680  (Precision: 85.36%, Count: 676)

Running for T = 60, X = 60...
--- Completed T = 60, X = 60 ---
  AUC:                    0.8674
  Mean Group Precision:   0.8415
  Buy Signal Thr

In [280]:
summary_table = pd.DataFrame(comparison_results)[[
    'T', 'AUC',
    'Mean_Group_Precision', 'Thresh_Buy', 'Buy_Tier_Precision', 
    'Thresh_StrongBuy', 'StrongBuy_Tier_Precision'
]]

print("==================================================")
print("FINAL MODEL COMPARISON ACROSS T SETUPS")
print("==================================================")
print(summary_table.to_string(index=False))

FINAL MODEL COMPARISON ACROSS T SETUPS
 T      AUC  Mean_Group_Precision  Thresh_Buy  Buy_Tier_Precision  Thresh_StrongBuy  StrongBuy_Tier_Precision
60 0.727247              0.720361       0.595            0.750000             0.815                  0.850806
60 0.758124              0.771429       0.400            0.757042             0.780                  0.850575
60 0.813747              0.815629       0.400            0.798830             0.680                  0.853550
60 0.867356              0.841480       0.400            0.814768             0.545                  0.850340


In [281]:
id_to_card_mapping = (
    df[['card_id', 'name', 'commercial_name']]
    .drop_duplicates('card_id')
    .set_index('card_id')
    .apply(lambda x: f"{x['name']} {x['commercial_name']}", axis=1)
    .to_dict()
)

In [282]:
SIM_START_DAY = 60
INITIAL_CAPITAL = 1000.0
SIM_END_DAY = 120     

YELLOW = "\033[93m"
GREEN = "\033[92m"
RED = "\033[91m"
CYAN = "\033[96m"
RESET = "\033[0m"

class DynamicMultiHorizonBacktester:
    def __init__(self, initial_capital, model, horizon_config, preferred_horizon=14, max_units_per_trade=3, slippage_pct=0.1, use_slippage=False, verbose=True):
        self.cash = initial_capital
        self.model = model
        self.horizon_config = horizon_config
        self.default_X = preferred_horizon
        self.max_units_per_trade = max_units_per_trade 
        self.slippage_pct = slippage_pct
        self.use_slippage = use_slippage
        self.verbose = verbose

        self.open_positions = []
        self.closed_trades = []
        self.daily_portfolio_log = []
        self.pnl_history = []

        print(horizon_config)

    def _get_signal(self, features_df):
        if len(features_df) == 0:
            return pd.DataFrame()
            
        model = self.model
        config = self.horizon_config
        
        cat_cols = ['seriesId', 'rarity']
        for col in cat_cols:
            if col in features_df.columns:
                features_df[col] = features_df[col].astype('category')

        feature_cols = [c for c in features_df.columns if c != 'card_id']
        probs = model.predict(features_df[feature_cols])
        
        res = pd.DataFrame({'card_id': features_df['card_id'], 'prob': probs})
        
        def assign_tier(p):
            if p >= config['strong_thresh']:
                return 'STRONG_BUY'
            elif p >= config['buy_thresh']:
                return 'BUY'
            return 'NO_BUY'
            
        res['signal'] = res['prob'].apply(assign_tier)
        return res

    def run_simulation(self, df_prices, SIM_START_DAY=45, SIM_END_DAY=90):
        if self.verbose:
            print(f"Starting Daily Dynamic Simulation (Days {SIM_START_DAY} to {SIM_END_DAY})...\n")

        for current_day in range(SIM_START_DAY, SIM_END_DAY + 1):
            if self.verbose:
                print(f"--- Day {current_day} ---")

            remaining_positions = []
            
            for pos in self.open_positions:
                cid = pos['card_id']
                cname = id_to_card_mapping.get(cid, str(cid))
                
                if current_day >= pos['maturity_day']:
                    price_today_series = df_prices[(df_prices['card_id'] == cid) & (df_prices['t'] == current_day)]['cardmarket_price']
                    
                    if len(price_today_series) == 0:
                        remaining_positions.append(pos)
                        continue
                        
                    current_price = price_today_series.values[0]

                    if self.use_slippage:
                        current_price *= (1 - self.slippage_pct)

                    df_sub = df_prices[df_prices['t'] <= current_day]
                    feat_today = extract_features_at_T(df_sub[df_sub['card_id'] == cid], T=current_day)
                    
                    eval_signal = self._get_signal(feat_today)
                    sig_type = eval_signal['signal'].values[0] if len(eval_signal) > 0 else 'NO_BUY'

                    if sig_type in ['BUY', 'STRONG_BUY'] and current_day < SIM_END_DAY:
                        pos['maturity_day'] = current_day + pos['horizon_X']
                        pos['last_evaluated_price'] = current_price
                        if self.verbose:
                            print(
                                f"{CYAN}Day {current_day}: HOLDING {pos['units']}x {cname} "
                                f"(Signal: {sig_type}) -> New Target Day {pos['maturity_day']}{RESET}"
                            )
                            
                        remaining_positions.append(pos)
                    else:
                        gross_proceeds = pos['units'] * current_price
                        pnl = gross_proceeds - pos['invested_usd']
                        self.pnl_history.append(pnl)
                        self.cash += gross_proceeds
                        color = GREEN if pnl >= 0 else RED

                        if self.verbose:
                            print(
                                f"{color}Day {current_day}: CLOSED {pos['units']}x {cname} "
                                f"(Signal: {sig_type}) | Bought: ${pos['buy_price']:.2f}, "
                                f"Sold: ${current_price:.2f} | PnL: ${pnl:+.2f}{RESET}"
                            )
                        
                        self.closed_trades.append({
                            'card_id': cid,
                            'entry_day': pos['entry_day'],
                            'exit_day': current_day,
                            'units': pos['units'],
                            'invested_usd': pos['invested_usd'],
                            'gross_proceeds': gross_proceeds,
                            'pnl': pnl,
                            'return_pct': (gross_proceeds / pos['invested_usd']) - 1.0
                        })
                else:
                    remaining_positions.append(pos)
                    
            self.open_positions = remaining_positions

            df_hist_today = df_prices[df_prices['t'] <= current_day]
            features_today = extract_features_at_T(df_hist_today, T=current_day)
            
            if len(features_today) > 0:
                signals_today = self._get_signal(features_today)
                buys_today = signals_today[signals_today['signal'] != 'NO_BUY'].copy()
                
                owned_card_ids = [p['card_id'] for p in self.open_positions]
                buys_today = buys_today[~buys_today['card_id'].isin(owned_card_ids)]
                
                prices_today = df_prices[df_prices['t'] == current_day].set_index('card_id')['cardmarket_price']

                for _, row in buys_today.iterrows():
                    cid = row['card_id']
                    sig = row['signal']
                    cname = id_to_card_mapping.get(cid, str(cid))

                    if cid not in prices_today:
                        continue
                        
                    buy_price = prices_today[cid]

                    if self.use_slippage:
                        buy_price *= (1 + self.slippage_pct)
                    
                    target_units = self.max_units_per_trade if sig == 'STRONG_BUY' else 1
                    
                    affordable_units = int(self.cash // buy_price)
                    units_to_buy = min(target_units, affordable_units)
                    
                    if units_to_buy < 1:
                        continue

                    cost_usd = units_to_buy * buy_price
                    self.cash -= cost_usd
                    if self.verbose:
                        print(
                            f"{YELLOW}Day {current_day}: BOUGHT {units_to_buy}x {cname} "
                            f"({sig}) @ ${buy_price:.2f} each | Total Cost: ${cost_usd:.2f}{RESET}"
                        )
                    
                    self.open_positions.append({
                        'card_id': cid,
                        'entry_day': current_day,
                        'maturity_day': current_day + self.default_X,
                        'horizon_X': self.default_X,
                        'buy_price': buy_price,
                        'units': units_to_buy,
                        'invested_usd': cost_usd
                    })

            prices_now = df_prices[df_prices['t'] == current_day].set_index('card_id')['cardmarket_price']
            current_inventory_value = sum([
                p['units'] * prices_now.get(p['card_id'], p['buy_price']) * (1 - self.slippage_pct if self.use_slippage else 1.0)
                for p in self.open_positions
            ])
                
            self.daily_portfolio_log.append({
                'day': current_day,
                'cash': self.cash,
                'inventory_value': current_inventory_value,
                'total_portfolio_value': self.cash + current_inventory_value,
                'open_positions_count': len(self.open_positions),
                'total_closed_trades': len(self.closed_trades)
            })

        return pd.DataFrame(self.daily_portfolio_log), pd.DataFrame(self.closed_trades), self.pnl_history

In [283]:
def run_monte_carlo_permutation(
    df_prices,
    backtester,
    n_iterations=500,
    sim_start_day=31,
    sim_end_day=90
):
    print(f"Running Monte Carlo Permutation ({n_iterations} runs)...")

    random_portfolio_returns = []

    for i in range(n_iterations):
        print(f"--- Monte Carlo Iteration {i + 1}/{n_iterations} ---")

        # Fresh backtester for every iteration
        bt = DynamicMultiHorizonBacktester(
            initial_capital=backtester.cash,
            model=backtester.model,
            horizon_config=backtester.horizon_config,
            preferred_horizon=backtester.default_X,
            max_units_per_trade=backtester.max_units_per_trade,
            slippage_pct=backtester.slippage_pct,
            use_slippage=backtester.use_slippage,
            verbose=False
        )

        def _random_signal_mock(features_df):
            if len(features_df) == 0:
                return pd.DataFrame()

            res = pd.DataFrame({
                'card_id': features_df['card_id'],
                'prob': np.random.uniform(0, 1, size=len(features_df))
            })

            res['signal'] = res['prob'].apply(
                lambda p:
                    'STRONG_BUY'
                    if p >= bt.horizon_config['strong_thresh']
                    else (
                        'BUY'
                        if p >= bt.horizon_config['buy_thresh']
                        else 'NO_BUY'
                    )
            )

            return res

        bt._get_signal = _random_signal_mock

        rand_log, rand_trades = bt.run_simulation(
            df_prices,
            sim_start_day,
            sim_end_day
        )

        final_value = rand_log['total_portfolio_value'].iloc[-1]

        print(
            f"Iteration {i + 1}: "
            f"Final Portfolio Value = ${final_value:.2f}, "
            f"Closed Trades: {len(rand_trades)}, "
            f"Open Positions: {rand_log['open_positions_count'].iloc[-1]}"
        )

        random_portfolio_returns.append(final_value)

    return random_portfolio_returns

In [284]:
def run_simulations(
    initial_capital, 
    model, horizon_config, 
    preferred_horizon=14, 
    max_units_per_trade=3, 
    slippage_pct=0.1, 
    use_slippage=False
):
    sim_engine = DynamicMultiHorizonBacktester(
        initial_capital=initial_capital,
        model=model,        
        horizon_config=horizon_config,      
        preferred_horizon=preferred_horizon,
        max_units_per_trade=max_units_per_trade,
        slippage_pct=slippage_pct,
        use_slippage=use_slippage
    )

    daily_log, closed_trades, pnl_history = sim_engine.run_simulation(
        df_holdout,
        SIM_START_DAY=SIM_START_DAY,
        SIM_END_DAY=SIM_END_DAY
    )   

    final_day_state = daily_log.iloc[-1]
    final_cash = final_day_state['cash']
    final_inventory_value = final_day_state['inventory_value']
    final_total_val = final_day_state['total_portfolio_value']
    net_pnl = final_total_val - INITIAL_CAPITAL
    total_return = (net_pnl / INITIAL_CAPITAL) * 100

    print("==================================================")
    print(f"     PORTFOLIO SUMMARY AT DAY {SIM_END_DAY}             ")
    print("==================================================")
    print(f"Starting Capital:            ${INITIAL_CAPITAL:,.2f}")
    print(f"Ending Cash Balance:         ${final_cash:,.2f}")
    print(f"Ending Held Inventory Value: ${final_inventory_value:,.2f}")
    print(f"--------------------------------------------------")
    print(f"Total Combined Portfolio:    ${final_total_val:,.2f}")
    print(f"Net Profit/Loss:             ${net_pnl:,.2f} ({total_return:+.2f}%)")
    print(f"Total Trades Realized:       {len(closed_trades)}")
    if len(closed_trades) > 0:
        win_rate = (closed_trades['pnl'] > 0).mean()
        print(f"Realized Win Rate:           {win_rate:.2%}")
    print(f"Cards Still Held at t=90:    {final_day_state['open_positions_count']}")
    print(f"Cars Sold by t=90:    {final_day_state['total_closed_trades']}")
    print("==================================================")

    return sim_engine, daily_log, closed_trades, pnl_history

In [285]:
OPTIMAL_X = 7
OPTIMAL_MODEL = models[(SIM_START_DAY, OPTIMAL_X)]
HORIZON_CONFIG = {
    "buy_thresh": summary_table[(summary_table['T'] == SIM_START_DAY)]['Thresh_Buy'].values[0],
    "strong_thresh": summary_table[(summary_table['T'] == SIM_START_DAY)]['Thresh_StrongBuy'].values[0]
}

print(f"Buy Threshold: {HORIZON_CONFIG['buy_thresh']:.3f}, Strong Buy Threshold: {HORIZON_CONFIG['strong_thresh']:.3f}")

sim_engine, daily_log, closed_trades, pnl_history = run_simulations(
    initial_capital=INITIAL_CAPITAL,
    model=OPTIMAL_MODEL,
    horizon_config=HORIZON_CONFIG,
    preferred_horizon=OPTIMAL_X,
    max_units_per_trade=1,
    slippage_pct=0.05,
    use_slippage=False
)

print(
    f"Number of positive PnL trades: {sum(1 for x in pnl_history if x > 0)} / {len(closed_trades)}"
)

print(
    f"Biggest single trade PnL: ${max(pnl_history):,.2f} "
)

print(
    f"Lowest single trade PnL: ${min(pnl_history):,.2f} "
)

print(
    f"Median positive PnL trade: ${np.median([x for x in pnl_history if x > 0]):,.2f} "
)

print(
    f"Median negative PnL trade: ${np.median([x for x in pnl_history if x < 0]):,.2f} "
)

Buy Threshold: 0.595, Strong Buy Threshold: 0.815
{'buy_thresh': np.float64(0.595), 'strong_thresh': np.float64(0.815)}
Starting Daily Dynamic Simulation (Days 60 to 120)...

--- Day 60 ---
Day 60: BOUGHT 1x Scalpereur EV01 (STRONG_BUY) @ $5.04 each | Total Cost: $5.04
Day 60: BOUGHT 1x Miraidon ex EV01 (STRONG_BUY) @ $2.40 each | Total Cost: $2.40
Day 60: BOUGHT 1x Koraidon ex EV01 (BUY) @ $2.06 each | Total Cost: $2.06
Day 60: BOUGHT 1x Gardevoir ex EV01 (BUY) @ $53.54 each | Total Cost: $53.54
Day 60: BOUGHT 1x Coiffeton EV02 (BUY) @ $7.82 each | Total Cost: $7.82
Day 60: BOUGHT 1x Cryodo EV02 (STRONG_BUY) @ $8.13 each | Total Cost: $8.13
Day 60: BOUGHT 1x Mesmérella EV02 (STRONG_BUY) @ $5.08 each | Total Cost: $5.08
Day 60: BOUGHT 1x Ferdeter EV02 (STRONG_BUY) @ $4.24 each | Total Cost: $4.24
Day 60: BOUGHT 1x Brome EV02 (BUY) @ $1.22 each | Total Cost: $1.22
Day 60: BOUGHT 1x Yuyu ex EV02 (STRONG_BUY) @ $28.89 each | Total Cost: $28.89
Day 60: BOUGHT 1x Palmaval ex EV02 (BUY) @ $4

In [286]:
OPTIMAL_X = 14
OPTIMAL_MODEL = models[(SIM_START_DAY, OPTIMAL_X)]
HORIZON_CONFIG = {
    "buy_thresh": summary_table[(summary_table['T'] == SIM_START_DAY)]['Thresh_Buy'].values[1],
    "strong_thresh": summary_table[(summary_table['T'] == SIM_START_DAY )]['Thresh_StrongBuy'].values[1]
}

print(f"Buy Threshold: {HORIZON_CONFIG['buy_thresh']:.3f}, Strong Buy Threshold: {HORIZON_CONFIG['strong_thresh']:.3f}")

sim_engine, daily_log, closed_trades, pnl_history = run_simulations(
    initial_capital=INITIAL_CAPITAL,
    model=OPTIMAL_MODEL,
    horizon_config=HORIZON_CONFIG,
    preferred_horizon=OPTIMAL_X,
    max_units_per_trade=1,
    slippage_pct=0.05,
    use_slippage=False
)

print(
    f"Number of positive PnL trades: {sum(1 for x in pnl_history if x > 0)} / {len(closed_trades)}"
)

print(
    f"Biggest single trade PnL: ${max(pnl_history):,.2f} "
)

print(
    f"Lowest single trade PnL: ${min(pnl_history):,.2f} "
)

print(
    f"Median positive PnL trade: ${np.median([x for x in pnl_history if x > 0]):,.2f} "
)

print(
    f"Median negative PnL trade: ${np.median([x for x in pnl_history if x < 0]):,.2f} "
)

Buy Threshold: 0.400, Strong Buy Threshold: 0.780
{'buy_thresh': np.float64(0.4), 'strong_thresh': np.float64(0.78)}
Starting Daily Dynamic Simulation (Days 60 to 120)...

--- Day 60 ---
Day 60: BOUGHT 1x Scalpereur EV01 (STRONG_BUY) @ $5.04 each | Total Cost: $5.04
Day 60: BOUGHT 1x Miraidon ex EV01 (STRONG_BUY) @ $2.40 each | Total Cost: $2.40
Day 60: BOUGHT 1x Koraidon ex EV01 (BUY) @ $2.06 each | Total Cost: $2.06
Day 60: BOUGHT 1x Coatox ex EV01 (BUY) @ $1.70 each | Total Cost: $1.70
Day 60: BOUGHT 1x Gardevoir ex EV01 (STRONG_BUY) @ $53.54 each | Total Cost: $53.54
Day 60: BOUGHT 1x Pepper EV01 (BUY) @ $4.23 each | Total Cost: $4.23
Day 60: BOUGHT 1x Coiffeton EV02 (BUY) @ $7.82 each | Total Cost: $7.82
Day 60: BOUGHT 1x Cryodo EV02 (BUY) @ $8.13 each | Total Cost: $8.13
Day 60: BOUGHT 1x Mesmérella EV02 (STRONG_BUY) @ $5.08 each | Total Cost: $5.08
Day 60: BOUGHT 1x Ferdeter EV02 (STRONG_BUY) @ $4.24 each | Total Cost: $4.24
Day 60: BOUGHT 1x Courrousinge ex EV02 (BUY) @ $2.62 e

In [287]:
OPTIMAL_X = 30
OPTIMAL_MODEL = models[(SIM_START_DAY, OPTIMAL_X)]
HORIZON_CONFIG = {
    "buy_thresh": summary_table[(summary_table['T'] == SIM_START_DAY)]['Thresh_Buy'].values[2],
    "strong_thresh": summary_table[(summary_table['T'] == SIM_START_DAY )]['Thresh_StrongBuy'].values[2]
}

print(f"Buy Threshold: {HORIZON_CONFIG['buy_thresh']:.3f}, Strong Buy Threshold: {HORIZON_CONFIG['strong_thresh']:.3f}")

sim_engine, daily_log, closed_trades, pnl_history = run_simulations(
    initial_capital=INITIAL_CAPITAL,
    model=OPTIMAL_MODEL,
    horizon_config=HORIZON_CONFIG,
    preferred_horizon=OPTIMAL_X,
    max_units_per_trade=1,
    slippage_pct=0.05,
    use_slippage=False
)

print(
    f"Number of positive PnL trades: {sum(1 for x in pnl_history if x > 0)} / {len(closed_trades)}"
)

print(
    f"Biggest single trade PnL: ${max(pnl_history):,.2f} "
)

print(
    f"Lowest single trade PnL: ${min(pnl_history):,.2f} "
)

print(
    f"Median positive PnL trade: ${np.median([x for x in pnl_history if x > 0]):,.2f} "
)

print(
    f"Median negative PnL trade: ${np.median([x for x in pnl_history if x < 0]):,.2f} "
)

Buy Threshold: 0.400, Strong Buy Threshold: 0.680
{'buy_thresh': np.float64(0.4), 'strong_thresh': np.float64(0.6799999999999999)}
Starting Daily Dynamic Simulation (Days 60 to 120)...

--- Day 60 ---
Day 60: BOUGHT 1x Scalpereur EV01 (STRONG_BUY) @ $5.04 each | Total Cost: $5.04
Day 60: BOUGHT 1x Miraidon ex EV01 (STRONG_BUY) @ $2.40 each | Total Cost: $2.40
Day 60: BOUGHT 1x Koraidon ex EV01 (BUY) @ $2.06 each | Total Cost: $2.06
Day 60: BOUGHT 1x Coatox ex EV01 (BUY) @ $1.70 each | Total Cost: $1.70
Day 60: BOUGHT 1x Fragroin ex EV01 (BUY) @ $1.47 each | Total Cost: $1.47
Day 60: BOUGHT 1x Gardevoir ex EV01 (STRONG_BUY) @ $53.54 each | Total Cost: $53.54
Day 60: BOUGHT 1x Pepper EV01 (STRONG_BUY) @ $4.23 each | Total Cost: $4.23
Day 60: BOUGHT 1x Chochodile EV02 (BUY) @ $20.18 each | Total Cost: $20.18
Day 60: BOUGHT 1x Coiffeton EV02 (STRONG_BUY) @ $7.82 each | Total Cost: $7.82
Day 60: BOUGHT 1x Cryodo EV02 (STRONG_BUY) @ $8.13 each | Total Cost: $8.13
Day 60: BOUGHT 1x Mesmérella

In [288]:
OPTIMAL_X = 60
OPTIMAL_MODEL = models[(SIM_START_DAY, OPTIMAL_X)]
HORIZON_CONFIG = {
    "buy_thresh": summary_table[(summary_table['T'] == SIM_START_DAY)]['Thresh_Buy'].values[3],
    "strong_thresh": summary_table[(summary_table['T'] == SIM_START_DAY )]['Thresh_StrongBuy'].values[3]
}

print(f"Buy Threshold: {HORIZON_CONFIG['buy_thresh']:.3f}, Strong Buy Threshold: {HORIZON_CONFIG['strong_thresh']:.3f}")

sim_engine, daily_log, closed_trades, pnl_history = run_simulations(
    initial_capital=INITIAL_CAPITAL,
    model=OPTIMAL_MODEL,
    horizon_config=HORIZON_CONFIG,
    preferred_horizon=OPTIMAL_X,
    max_units_per_trade=1,
    slippage_pct=0.05,
    use_slippage=False
)

print(
    f"Number of positive PnL trades: {sum(1 for x in pnl_history if x > 0)} / {len(closed_trades)}"
)

print(
    f"Biggest single trade PnL: ${max(pnl_history):,.2f} "
)

print(
    f"Lowest single trade PnL: ${min(pnl_history):,.2f} "
)

print(
    f"Median positive PnL trade: ${np.median([x for x in pnl_history if x > 0]):,.2f} "
)

print(
    f"Median negative PnL trade: ${np.median([x for x in pnl_history if x < 0]):,.2f} "
)

Buy Threshold: 0.400, Strong Buy Threshold: 0.545
{'buy_thresh': np.float64(0.4), 'strong_thresh': np.float64(0.545)}
Starting Daily Dynamic Simulation (Days 60 to 120)...

--- Day 60 ---
Day 60: BOUGHT 1x Scalpereur EV01 (STRONG_BUY) @ $5.04 each | Total Cost: $5.04
Day 60: BOUGHT 1x Miraidon ex EV01 (BUY) @ $2.40 each | Total Cost: $2.40
Day 60: BOUGHT 1x Gardevoir ex EV01 (STRONG_BUY) @ $53.54 each | Total Cost: $53.54
Day 60: BOUGHT 1x Pepper EV01 (STRONG_BUY) @ $4.23 each | Total Cost: $4.23
Day 60: BOUGHT 1x Chochodile EV02 (STRONG_BUY) @ $20.18 each | Total Cost: $20.18
Day 60: BOUGHT 1x Coiffeton EV02 (STRONG_BUY) @ $7.82 each | Total Cost: $7.82
Day 60: BOUGHT 1x Cryodo EV02 (STRONG_BUY) @ $8.13 each | Total Cost: $8.13
Day 60: BOUGHT 1x Mesmérella EV02 (STRONG_BUY) @ $5.08 each | Total Cost: $5.08
Day 60: BOUGHT 1x Simularbre EV02 (STRONG_BUY) @ $16.53 each | Total Cost: $16.53
Day 60: BOUGHT 1x Ferdeter EV02 (STRONG_BUY) @ $4.24 each | Total Cost: $4.24
Day 60: BOUGHT 1x Cou

In [289]:
model_14 = models[(SIM_START_DAY, 14)]
model_30 = models[(SIM_START_DAY, 30)]
model_60 = models[(SIM_START_DAY, 60)]

model_14.save_model("models/model_14.txt")
model_30.save_model("models/model_30.txt")
model_60.save_model("models/model_60.txt")